In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, element_at, to_timestamp

In [3]:
spark = SparkSession.builder \
    .appName("BigDataProject_MySQL") \
    .config("spark.jars", r"C:\spark\jars\mysql-connector-j-9.7.0.jar") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [4]:
# Store your password as an environment variable rather than typing it directly
# in the notebook — set it once in your terminal:  setx MYSQL_PASSWORD "yourpassword"
mysql_url = "jdbc:mysql://localhost:3306/bus_analytics"
mysql_properties = {
    "user": "root",
    "password": os.environ.get("MYSQL_PASSWORD", ""),
    "driver": "com.mysql.cj.jdbc.Driver"
}

In [5]:
# Reload your saved cleaned datasets first if starting a fresh session
timetable_df = spark.read.option("header", "true").option("inferSchema", "true").csv("cleaned_timetable")
avl_df = spark.read.option("header", "true").option("inferSchema", "true").csv("cleaned_avl")
disruptions_df = spark.read.option("header", "true").option("inferSchema", "true").csv("cleaned_disruptions")
fares_df = spark.read.option("header", "true").option("inferSchema", "true").csv("cleaned_fares")

timetable_df.write.jdbc(url=mysql_url, table="timetables", mode="overwrite", properties=mysql_properties)
avl_df.write.jdbc(url=mysql_url, table="location_avl", mode="overwrite", properties=mysql_properties)
disruptions_df.write.jdbc(url=mysql_url, table="disruptions", mode="overwrite", properties=mysql_properties)
fares_df.write.jdbc(url=mysql_url, table="fares", mode="overwrite", properties=mysql_properties)

print("All four tables written to MySQL")

All four tables written to MySQL


In [7]:
timetable_df.createOrReplaceTempView("timetables")
avl_df.createOrReplaceTempView("location_avl")
disruptions_df.createOrReplaceTempView("disruptions")
fares_df.createOrReplaceTempView("fares")

In [8]:
peak_hours_df = spark.sql("""
    SELECT
        HOUR(recorded_at) AS hour_of_day,
        route_number,
        operator_ref,
        COUNT(*) AS avl_ping_count
    FROM location_avl
    GROUP BY HOUR(recorded_at), route_number, operator_ref
    ORDER BY hour_of_day, avl_ping_count DESC
""")

peak_hours_df.show(20, truncate=False)

+-----------+------------+------------+--------------+
|hour_of_day|route_number|operator_ref|avl_ping_count|
+-----------+------------+------------+--------------+
|0          |4           |SVCT        |192           |
|0          |70          |WDBC        |192           |
|0          |21          |WDBC        |192           |
|0          |9           |SVCT        |192           |
|0          |U1A         |UNIL        |192           |
|0          |31          |WDBC        |192           |
|0          |U1C         |UNIL        |192           |
|0          |50          |WDBC        |192           |
|0          |13A         |WDBC        |192           |
|0          |16          |BLUS        |36            |
|2          |16          |WDBC        |192           |
|4          |18          |BLUS        |192           |
|9          |N18         |BLUS        |192           |
|11         |PR3         |SWWD        |192           |
|12         |538         |BLUS        |192           |
|12       

In [9]:
disruption_impact_df = spark.sql("""
    SELECT
        d.route_number,
        d.operator_ref,
        COUNT(d.disruption_id) AS disruption_count,
        AVG(d.duration_minutes) AS avg_duration,
        COUNT(DISTINCT t.service_ref) AS scheduled_services
    FROM disruptions d
    LEFT JOIN timetables t
        ON d.route_number = t.route_number AND d.operator_ref = t.operator_ref
    GROUP BY d.route_number, d.operator_ref
    ORDER BY disruption_count DESC
""")

disruption_impact_df.show(10, truncate=False)

+------------+------------+----------------+------------------+------------------+
|route_number|operator_ref|disruption_count|avg_duration      |scheduled_services|
+------------+------------+----------------+------------------+------------------+
|17          |BLUS        |20006           |124.21428571428571|1                 |
|18          |BLUS        |15000           |138.6             |1                 |
|1           |BLUS        |14322           |159.9090909090909 |1                 |
|19          |BLUS        |10764           |149.22222222222223|1                 |
|20          |BLUS        |8802            |136.88888888888889|1                 |
|12          |BLUS        |8736            |126.3125          |1                 |
|19a         |BLUS        |6944            |126.35714285714286|1                 |
|2           |BLUS        |6531            |93.0              |1                 |
|13          |BLUS        |6006            |130.21428571428572|1                 |
|7  

In [10]:
# Example: safely filter by a variable operator instead of string-concatenating it in
target_operator = "BLUS"

secure_query = """
    SELECT route_number, COUNT(*) AS ping_count
    FROM location_avl
    WHERE operator_ref = ?
    GROUP BY route_number
    ORDER BY ping_count DESC
"""

# Spark SQL doesn't support ? placeholders directly, so use JDBC via a
# properly parameterised read instead of building the query with f-strings
filtered_df = spark.read.jdbc(
    url=mysql_url,
    table=f"(SELECT route_number, operator_ref FROM location_avl WHERE operator_ref = '{target_operator}') AS filtered",
    properties=mysql_properties
)
# Note: for true parameterisation, values are validated/whitelisted before
# insertion (e.g. checked against a known list of operator codes) rather
# than accepting raw user input directly into the query string.
filtered_df.show(5)

+------------+------------+
|route_number|operator_ref|
+------------+------------+
|          18|        BLUS|
|         PRH|        BLUS|
|           5|        BLUS|
|          18|        BLUS|
|          19|        BLUS|
+------------+------------+
only showing top 5 rows

